# AML Rules Engine -- Transaction-Level Risk Scoring
---
Applies **50 deterministic rules** across 10 categories on top of the typology
detector output. Each transaction receives individual rule flags (0/1), a severity-
weighted composite score, and an alert level.

### Rule Categories (10 groups, 50 rules)
| # | Category | Rules | Severity Mix |
|---|----------|-------|--------------|
| 1 | Amount | 5 | 2 High, 1 Medium, 2 Low |
| 2 | Time | 3 | 1 High, 2 Medium |
| 3 | Account Velocity | 7 | 3 High, 4 Medium |
| 4 | Customer Velocity | 4 | 1 High, 3 Medium |
| 5 | KYC / Account | 7 | 4 High, 3 Medium |
| 6 | Beneficiary | 4 | 2 High, 2 Medium |
| 7 | Device / Channel | 6 | 1 High, 4 Medium, 1 Low |
| 8 | Structuring | 2 | 2 High |
| 9 | Occupation / Industry | 4 | 0 High, 2 Medium, 2 Low |
| 10 | Combined Risk Combos | 8 | 7 High, 1 Medium |

### Scoring
- Each triggered rule adds its severity (1/2/3) to the transaction's composite score
- Alert levels: **Critical** (score >= 9), **High** (>= 6), **Medium** (>= 3), **Low** (>= 1), **None** (0)

### Input / Output
- **Input**: `transactions_labeled.csv` from the Typology Detector
- **Output**: Same data + 50 rule columns + `rules_triggered`, `rule_score`, `alert_level`


## 1 -- Environment Setup


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
import warnings, os

warnings.filterwarnings('ignore')
OUTPUT_DIR = "../outputs_updated"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 -- Load Typology Detector Output


In [2]:
# ============================================================
# UPDATE THIS PATH to your labeled transaction file
# ============================================================
INPUT_FILE = "../outputs_updated/stg_transactions_typology.parquet"

# Fallback paths
if not os.path.exists(INPUT_FILE):
    for alt in ["../outputs_updated/stg_transactions_typology.csv",
                "stg_transactions_typology.parquet",
                "stg_transactions_typology.csv"]:
        if os.path.exists(alt):
            INPUT_FILE = alt
            break

ext = INPUT_FILE.rsplit('.', 1)[-1].lower()
if ext == 'parquet':
    df = pd.read_parquet(INPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE, low_memory=False)

print(f"Loaded: {INPUT_FILE}")
print(f"  {len(df):,} rows x {len(df.columns)} columns")

# Show existing typology labels if present
if "is_aml" in df.columns:
    aml_count = (df["is_aml"] == 1).sum()
    print(f"  Pre-labeled AML transactions: {aml_count:,} ({aml_count/len(df)*100:.2f}%)")
if "aml_typology" in df.columns:
    typs = df[df["aml_typology"].notna() & (df["aml_typology"] != "")]
    if len(typs) > 0:
        print(f"  Typologies present: {typs['aml_typology'].nunique()}")

Loaded: ../outputs_updated/stg_transactions_typology.parquet
  327,025 rows x 97 columns
  Pre-labeled AML transactions: 22,326 (6.83%)
  Typologies present: 12


## 3 -- Column Resolution
Flexibly resolve column names from both the bank's raw format and the
synthetic generator's clean format.


In [3]:
def find_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Resolve all needed columns
COL = {
    "amount":       find_col(["transaction_amount","amount","Transaction Amount"]),
    "cash_flag":    find_col(["cash_flag","Cash Flag"]),
    "timestamp":    find_col(["timestamp","Timestamp"]),
    "datestamp":     find_col(["datestamp","Datestamp"]),
    "txn_type":     find_col(["transaction_type_dr_cr","txn_type","Transaction Type"]),
    "channel":      find_col(["transaction_mode_channel_bank","channel_bank","Transaction Mode/Channel - Bank"]),
    "status":       find_col(["transaction_status","txn_status","Transaction Status"]),
    "acct":         find_col(["customer_account_number","account_number","Customer Account Number"]),
    "cp_acct":      find_col(["counterparty_account_number","cp_account_number","Counterparty Account Number"]),
    "cif":          find_col(["customer_cif_id","cif_id","Customer CIF/ID Number"]),
    "acct_status":  find_col(["account_wallet_status","account_status","Account/Wallet Status"]),
    "acct_open":    find_col(["account_wallet_opening_date","account_open_date","Account/Wallet Opening Date"]),
    "risk_score":   find_col(["customer_current_risk_score","risk_score","Customer Current Risk Score"]),
    "pep_flag":     find_col(["pep_flag","PEP Flag"]),
    "vpn_flag":     find_col(["vpn_flag","VPN Flag"]),
    "emulator_flag":find_col(["emulator_flag","Emulator Flag"]),
    "cust_type":    find_col(["customer_type","Customer Type"]),
    "acct_category":find_col(["account_category","Account Category"]),
    "acct_type":    find_col(["account_type","Account Type"]),
    "occupation":   find_col(["customer_occupation_industry","occupation","Customer Occupation/Industry"]),
    "annual_income":find_col(["annual_income","Annual Income"]),
    "sender_cc":    find_col(["sender_country_code","sender_country","Sender Country Code*"]),
    "receiver_cc":  find_col(["receiver_country_code","receiver_country","Receiver Country Code*"]),
    "wallet_kyc":   find_col(["wallet_kyc_category","wallet_kyc","Wallet KYC Category"]),
    "device_id":    find_col(["device_id_fingerprint","device_id","Device ID/Fingerprint"]),
    "credit_sum":   find_col(["credit_summation_period","credit_sum_period","Credit Summation of the account for the period"]),
    "debit_sum":    find_col(["debit_summation_period","debit_sum_period","Debit Summation of the account for the period"]),
}

print("Column resolution:")
resolved = sum(1 for v in COL.values() if v)
print(f"  Resolved: {resolved}/{len(COL)}")
missing = [k for k, v in COL.items() if not v]
if missing:
    print(f"  Missing (will use safe defaults): {missing}")



Column resolution:
  Resolved: 27/27


## 4 -- Build Derived Features
Compute all intermediate fields needed by the rules.


In [4]:
print("Building derived features...")

# -- Core numeric/string fields --
df["_amt"] = pd.to_numeric(df[COL["amount"]], errors="coerce").fillna(0) if COL["amount"] else 0
df["_cash"] = df[COL["cash_flag"]].astype(str).str.strip().str.upper().isin(["Y","1","YES","TRUE"]) if COL["cash_flag"] else False

# -- Datetime --
def parse_dt(row):
    ds = str(row.get(COL["datestamp"], "") if COL["datestamp"] else "")
    ts = str(row.get(COL["timestamp"], "00:00:00") if COL["timestamp"] else "00:00:00")
    for fmt in ["%d-%m-%Y %H:%M:%S","%d/%m/%Y %H:%M:%S","%Y-%m-%d %H:%M:%S"]:
        try:
            return datetime.strptime(f"{ds} {ts}".strip(), fmt)
        except:
            continue
    return pd.NaT

df["_dt"] = df.apply(parse_dt, axis=1)
df["_hour"] = df["_dt"].apply(lambda x: x.hour if pd.notna(x) else 12)
df["_dow"] = df["_dt"].apply(lambda x: x.weekday() if pd.notna(x) else 0)  # 0=Mon, 6=Sun

# -- Time-based flags --
df["_is_night"] = df["_hour"].apply(lambda h: h >= 22 or h < 6)
df["_is_weekend"] = df["_dow"].isin([5, 6])

# -- Round amount --
df["_is_round"] = df["_amt"].apply(lambda x: x > 0 and x == int(x) and int(x) % 1000 == 0)

# -- Account open days --
def parse_date_col(col_name):
    if not col_name or col_name not in df.columns:
        return pd.NaT
    return pd.to_datetime(df[col_name], format="%d-%m-%Y", errors="coerce")

acct_open_dates = parse_date_col(COL["acct_open"])
if acct_open_dates is not None and not isinstance(acct_open_dates, type(pd.NaT)):
    df["_acct_open_days"] = (df["_dt"] - acct_open_dates).dt.days.fillna(9999)
else:
    df["_acct_open_days"] = 9999

# -- String flags --
def flag_col(col_key):
    if COL[col_key] and COL[col_key] in df.columns:
        return df[COL[col_key]].astype(str).str.strip().str.upper().isin(["Y","1","YES","TRUE"])
    return pd.Series(False, index=df.index)

df["_pep"] = flag_col("pep_flag")
df["_vpn"] = flag_col("vpn_flag")
df["_emulator"] = flag_col("emulator_flag")

# -- Channel --
df["_channel"] = df[COL["channel"]].astype(str).str.strip().str.upper() if COL["channel"] else ""

# -- Risk score --
df["_risk"] = df[COL["risk_score"]].astype(str).str.strip().str.upper() if COL["risk_score"] else ""

# -- Occupation --
df["_occupation"] = df[COL["occupation"]].astype(str).str.strip().str.lower() if COL["occupation"] else ""

# -- Income --
df["_income"] = pd.to_numeric(df[COL["annual_income"]], errors="coerce").fillna(0) if COL["annual_income"] else 0

# -- Customer type / Account type --
df["_cust_type"] = df[COL["cust_type"]].astype(str).str.strip().str.lower() if COL["cust_type"] else ""
df["_acct_cat"] = df[COL["acct_category"]].astype(str).str.strip().str.lower() if COL["acct_category"] else ""
df["_acct_type"] = df[COL["acct_type"]].astype(str).str.strip().str.lower() if COL["acct_type"] else ""

# -- Account / CIF identifiers --
df["_acct"] = df[COL["acct"]].astype(str).str.strip() if COL["acct"] else ""
df["_cif"] = df[COL["cif"]].astype(str).str.strip() if COL["cif"] else df["_acct"]

# -- Country codes --
df["_sender_cc"] = df[COL["sender_cc"]].astype(str).str.strip().str.upper() if COL["sender_cc"] else "IN"
df["_receiver_cc"] = df[COL["receiver_cc"]].astype(str).str.strip().str.upper() if COL["receiver_cc"] else "IN"

# -- Wallet KYC level --
df["_wallet_kyc"] = df[COL["wallet_kyc"]].astype(str).str.strip().str.lower() if COL["wallet_kyc"] else ""

# -- Balance proxy (credit_sum as rough balance) --
df["_balance_proxy"] = pd.to_numeric(df[COL["credit_sum"]], errors="coerce").fillna(1) if COL["credit_sum"] else 1
df["_balance_proxy"] = df["_balance_proxy"].replace(0, 1)  # avoid division by zero

# -- FATF high-risk country list --
FATF_HIGH_RISK = {"KP","IR","MM","AF","SY","YE","HT","SD","ML","BF","MZ","TZ","CD","SS","LY",
                   "PK","NI","JM","TR","PH","AE"}

df["_sender_high_risk_cc"] = df["_sender_cc"].isin(FATF_HIGH_RISK)
df["_receiver_high_risk_cc"] = df["_receiver_cc"].isin(FATF_HIGH_RISK)

# -- Beneficiary type heuristic (offshore = foreign receiver, crypto = not applicable in most bank data) --
df["_is_offshore"] = (df["_receiver_cc"] != "IN") & (df["_receiver_cc"] != "") & (df["_receiver_cc"] != "NAN")

print(f"  Derived features built: {sum(1 for c in df.columns if c.startswith('_')):,} working columns")



Building derived features...
  Derived features built: 28 working columns


## 5 -- Compute Velocity & Aggregation Features
Rolling window counts and volumes per account and per customer.


In [5]:
print("Computing velocity features (this may take a minute)...")

# Sort by account + datetime for rolling calculations
df = df.sort_values(["_acct", "_dt"]).reset_index(drop=True)

# -- Account-level velocity --
# For each transaction, count/sum prior txns from same account within window
def compute_velocity(group_col, prefix):
    # group_col: column to group by (_acct or _cif)
    # prefix: column name prefix
    counts_1hr = []
    counts_24hr = []
    counts_7d = []
    sums_24hr = []
    sums_7d = []

    grouped = df.groupby(group_col)
    total_groups = len(grouped)
    processed = 0

    for gname, gdf in grouped:
        processed += 1
        if processed % 3000 == 0:
            print(f"    {prefix}: {processed:,}/{total_groups:,} groups...")

        dts = gdf["_dt"].values
        amts = gdf["_amt"].values
        n = len(gdf)

        c1 = np.zeros(n, dtype=int)
        c24 = np.zeros(n, dtype=int)
        c7 = np.zeros(n, dtype=int)
        s24 = np.zeros(n, dtype=float)
        s7 = np.zeros(n, dtype=float)

        for i in range(n):
            t = dts[i]
            if pd.isna(t):
                continue
            for j in range(i - 1, -1, -1):
                if pd.isna(dts[j]):
                    continue
                diff = (t - dts[j]) / np.timedelta64(1, 'h')
                if diff > 168:  # > 7 days in hours
                    break
                if diff <= 1:
                    c1[i] += 1
                if diff <= 24:
                    c24[i] += 1
                    s24[i] += amts[j]
                if diff <= 168:
                    c7[i] += 1
                    s7[i] += amts[j]

        counts_1hr.extend(c1)
        counts_24hr.extend(c24)
        counts_7d.extend(c7)
        sums_24hr.extend(s24)
        sums_7d.extend(s7)

    df[f"{prefix}_count_1hr"] = counts_1hr
    df[f"{prefix}_count_24hr"] = counts_24hr
    df[f"{prefix}_count_7d"] = counts_7d
    df[f"{prefix}_sum_24hr"] = sums_24hr
    df[f"{prefix}_sum_7d"] = sums_7d

compute_velocity("_acct", "_acct_vel")
compute_velocity("_cif", "_cust_vel")

# -- Dormancy flag: gap > 30 days since last txn from same account --
print("  Computing dormancy flags...")
df["_prev_txn_gap_days"] = df.groupby("_acct")["_dt"].diff().dt.days.fillna(0)
df["_dormancy"] = df["_prev_txn_gap_days"] > 30

# -- Amount z-score (30-day rolling mean/std per account) --
print("  Computing 30-day amount z-scores...")
zscore_vals = np.zeros(len(df))
for acct, gdf in df.groupby("_acct"):
    idxs = gdf.index.values
    dts = gdf["_dt"].values
    amts = gdf["_amt"].values
    for i in range(len(idxs)):
        t = dts[i]
        if pd.isna(t):
            continue
        window_amts = []
        for j in range(i - 1, -1, -1):
            if pd.isna(dts[j]):
                continue
            diff_days = (t - dts[j]) / np.timedelta64(1, 'D')
            if diff_days > 30:
                break
            window_amts.append(amts[j])
        if len(window_amts) >= 3:
            mu = np.mean(window_amts)
            sigma = np.std(window_amts)
            if sigma > 0:
                zscore_vals[idxs[i]] = (amts[i] - mu) / sigma

df["_amt_zscore_30d"] = zscore_vals

# -- Amount to balance ratio --
df["_amt_to_balance"] = df["_amt"] / df["_balance_proxy"]

print("  Velocity features complete")



Computing velocity features (this may take a minute)...
    _acct_vel: 3,000/4,766 groups...
    _cust_vel: 3,000/3,291 groups...
  Computing dormancy flags...
  Computing 30-day amount z-scores...
  Velocity features complete


## 6 -- Rule Definitions
All 50 rules defined as a registry. Each rule has a name, group, severity (1-3),
and a lambda condition that returns a boolean Series.


In [6]:
# ============================================================
# RULE REGISTRY -- 50 rules across 10 groups
# Each: (column_name, group, readable_name, severity, condition_fn)
# ============================================================

RULES = [
    # --- GROUP 1: Amount (5 rules) ---
    ("rule_large_cash_deposit", "Amount", "Large Cash Deposit", 3,
     lambda d: d["_cash"] & (d["_amt"] > 50000)),

    ("rule_cash_just_below_threshold", "Amount", "Cash Just Below Threshold", 3,
     lambda d: d["_cash"] & (d["_amt"] >= 8000) & (d["_amt"] <= 9999)),

    ("rule_high_value_transfer", "Amount", "High Value Transfer", 2,
     lambda d: d["_amt"] > 100000),

    ("rule_micro_transaction", "Amount", "Micro Transaction", 1,
     lambda d: (d["_amt"] > 0) & (d["_amt"] < 10)),

    ("rule_round_amount_large", "Amount", "Round Amount Large", 1,
     lambda d: d["_is_round"] & (d["_amt"] >= 10000)),

    # --- GROUP 2: Time (3 rules) ---
    ("rule_night_transaction", "Time", "Night Transaction", 2,
     lambda d: d["_is_night"]),

    ("rule_weekend_high_value", "Time", "Weekend High Value", 2,
     lambda d: d["_is_weekend"] & (d["_amt"] > 20000)),

    ("rule_dormant_account_activation", "Time", "Dormant Account Activation", 3,
     lambda d: d["_dormancy"] & (d["_amt"] > 5000)),

    # --- GROUP 3: Account Velocity (7 rules) ---
    ("rule_high_acct_velocity_1hr", "Account Velocity", "High Acct Velocity 1hr", 3,
     lambda d: d["_acct_vel_count_1hr"] > 5),

    ("rule_rapid_burst", "Account Velocity", "Rapid Burst", 3,
     lambda d: d["_acct_vel_count_1hr"] > 3),

    ("rule_high_acct_velocity_24hr", "Account Velocity", "High Acct Velocity 24hr", 2,
     lambda d: d["_acct_vel_count_24hr"] > 20),

    ("rule_high_acct_velocity_7d", "Account Velocity", "High Acct Velocity 7d", 2,
     lambda d: d["_acct_vel_count_7d"] > 60),

    ("rule_high_acct_volume_24hr", "Account Velocity", "High Acct Volume 24hr", 2,
     lambda d: d["_acct_vel_sum_24hr"] > 500000),

    ("rule_high_acct_volume_7d", "Account Velocity", "High Acct Volume 7d", 2,
     lambda d: d["_acct_vel_sum_7d"] > 2000000),

    ("rule_amount_spike_30d", "Account Velocity", "Amount Spike 30d", 3,
     lambda d: d["_amt_zscore_30d"] > 4),

    # --- GROUP 4: Customer Velocity (4 rules) ---
    ("rule_high_cust_velocity_1hr", "Customer Velocity", "High Cust Velocity 1hr", 3,
     lambda d: d["_cust_vel_count_1hr"] > 8),

    ("rule_high_cust_velocity_24hr", "Customer Velocity", "High Cust Velocity 24hr", 2,
     lambda d: d["_cust_vel_count_24hr"] > 30),

    ("rule_high_cust_volume_24hr", "Customer Velocity", "High Cust Volume 24hr", 2,
     lambda d: d["_cust_vel_sum_24hr"] > 1000000),

    ("rule_high_cust_volume_7d", "Customer Velocity", "High Cust Volume 7d", 2,
     lambda d: d["_cust_vel_sum_7d"] > 3000000),

    # --- GROUP 5: KYC / Account (7 rules) ---
    ("rule_low_kyc_high_amount", "KYC / Account", "Low KYC High Amount", 3,
     lambda d: (d["_wallet_kyc"].str.contains("min", case=False, na=False)) & (d["_amt"] > 30000)),

    ("rule_new_account_large_txn", "KYC / Account", "New Account Large Txn", 3,
     lambda d: (d["_acct_open_days"] < 60) & (d["_amt"] > 10000)),

    ("rule_high_amount_to_balance", "KYC / Account", "High Amount to Balance", 2,
     lambda d: d["_amt_to_balance"] > 3),

    ("rule_very_high_risk_customer", "KYC / Account", "Very High Risk Customer", 3,
     lambda d: d["_risk"].isin(["HIGH","VERY_HIGH","VERY HIGH","3"])),

    ("rule_pep_high_value", "KYC / Account", "PEP High Value", 3,
     lambda d: d["_pep"] & (d["_amt"] > 10000)),

    ("rule_high_risk_country_sender", "KYC / Account", "High Risk Country Sender", 2,
     lambda d: d["_sender_high_risk_cc"]),

    ("rule_corporate_large_cash", "KYC / Account", "Corporate Large Cash", 2,
     lambda d: (d["_cust_type"].str.contains("non-individual|corporate|business", case=False, na=False)
                | d["_acct_cat"].str.contains("current", case=False, na=False))
               & d["_cash"] & (d["_amt"] > 20000)),

    # --- GROUP 6: Beneficiary (4 rules) ---
    ("rule_high_risk_beneficiary", "Beneficiary", "High Risk Beneficiary", 3,
     lambda d: d["_receiver_high_risk_cc"]),

    ("rule_crypto_transfer", "Beneficiary", "Crypto Transfer", 2,
     lambda d: pd.Series(False, index=d.index)),  # placeholder: bank data rarely has crypto flag

    ("rule_offshore_transfer", "Beneficiary", "Offshore Transfer", 3,
     lambda d: d["_is_offshore"]),

    ("rule_high_risk_bene_country", "Beneficiary", "High Risk Bene Country", 2,
     lambda d: d["_receiver_high_risk_cc"]),

    # --- GROUP 7: Device / Channel (6 rules) ---
    ("rule_rooted_device", "Device / Channel", "Rooted Device", 2,
     lambda d: pd.Series(False, index=d.index)),  # placeholder: needs rooted_flag column

    ("rule_vpn_proxy_detected", "Device / Channel", "VPN / Proxy Detected", 2,
     lambda d: d["_vpn"]),

    ("rule_emulator_detected", "Device / Channel", "Emulator Detected", 3,
     lambda d: d["_emulator"]),

    ("rule_new_device_large_txn", "Device / Channel", "New Device Large Txn", 1,
     lambda d: pd.Series(False, index=d.index)),  # placeholder: needs device_first_seen

    ("rule_atm_high_withdrawal", "Device / Channel", "ATM High Withdrawal", 2,
     lambda d: (d["_channel"].str.contains("ATM", case=False, na=False)) & (d["_amt"] > 20000)),

    ("rule_branch_night_txn", "Device / Channel", "Branch Night Txn", 2,
     lambda d: (d["_channel"].str.contains("BRANCH", case=False, na=False)) & d["_is_night"]),

    # --- GROUP 8: Structuring (2 rules) ---
    ("rule_structuring_pattern", "Structuring", "Structuring Pattern", 3,
     lambda d: d["_cash"] & (d["_amt"] >= 8500) & (d["_amt"] <= 9999)),

    ("rule_multiple_small_cash", "Structuring", "Multiple Small Cash", 3,
     lambda d: d["_cash"] & (d["_acct_vel_count_7d"] > 5) & (d["_amt"] < 10000)),

    # --- GROUP 9: Occupation / Industry (4 rules) ---
    ("rule_high_risk_industry", "Occupation / Industry", "High Risk Industry", 1,
     lambda d: d["_occupation"].str.contains("real estate|construction|unknown", case=False, na=False)),

    ("rule_student_high_value", "Occupation / Industry", "Student High Value", 2,
     lambda d: (d["_occupation"].str.contains("student", case=False, na=False)) & (d["_amt"] > 50000)),

    ("rule_unemployed_large_transfer", "Occupation / Industry", "Unemployed Large Transfer", 2,
     lambda d: (d["_occupation"].str.contains("unemployed|retired", case=False, na=False)) & (d["_amt"] > 20000)),

    ("rule_freelancer_offshore", "Occupation / Industry", "Freelancer Offshore", 2,
     lambda d: (d["_occupation"].str.contains("freelance", case=False, na=False)) & d["_is_offshore"]),

    # --- GROUP 10: Combined Risk Combos (8 rules) ---
    ("rule_pep_crypto_transfer", "Combined Combos", "PEP + Crypto Transfer", 3,
     lambda d: pd.Series(False, index=d.index)),  # placeholder

    ("rule_very_high_risk_offshore", "Combined Combos", "Very High Risk + Offshore", 3,
     lambda d: d["_risk"].isin(["HIGH","VERY_HIGH","VERY HIGH","3"]) & d["_is_offshore"]),

    ("rule_low_kyc_offshore", "Combined Combos", "Low KYC + Offshore", 3,
     lambda d: (d["_wallet_kyc"].str.contains("min", case=False, na=False)) & d["_is_offshore"]),

    ("rule_low_income_large_txn", "Combined Combos", "Low Income Large Txn", 2,
     lambda d: (d["_income"] > 0) & (d["_income"] < 300000) & (d["_amt"] > 50000)),

    ("rule_vpn_offshore", "Combined Combos", "VPN + Offshore", 3,
     lambda d: d["_vpn"] & d["_is_offshore"]),

    ("rule_emulator_crypto", "Combined Combos", "Emulator + Crypto", 3,
     lambda d: pd.Series(False, index=d.index)),  # placeholder

    ("rule_new_account_offshore", "Combined Combos", "New Account + Offshore", 3,
     lambda d: (d["_acct_open_days"] < 90) & d["_is_offshore"]),

    ("rule_new_acct_high_cust_velocity", "Combined Combos", "New Acct + High Velocity", 3,
     lambda d: (d["_acct_open_days"] < 90) & (d["_cust_vel_count_24hr"] > 10)),
]

print(f"Rules defined: {len(RULES)}")
sev_counts = {1: 0, 2: 0, 3: 0}
grp_counts = defaultdict(int)
for _, grp, _, sev, _ in RULES:
    sev_counts[sev] += 1
    grp_counts[grp] += 1

print(f"  Severity 3 (High):   {sev_counts[3]}")
print(f"  Severity 2 (Medium): {sev_counts[2]}")
print(f"  Severity 1 (Low):    {sev_counts[1]}")
print(f"\nBy group:")
for g in sorted(grp_counts):
    print(f"  {g:<25s} {grp_counts[g]:>3} rules")



Rules defined: 50
  Severity 3 (High):   23
  Severity 2 (Medium): 23
  Severity 1 (Low):    4

By group:
  Account Velocity            7 rules
  Amount                      5 rules
  Beneficiary                 4 rules
  Combined Combos             8 rules
  Customer Velocity           4 rules
  Device / Channel            6 rules
  KYC / Account               7 rules
  Occupation / Industry       4 rules
  Structuring                 2 rules
  Time                        3 rules


## 7 -- Execute All Rules


In [7]:
print("Executing all 50 rules...")

rule_columns = []
severity_map = {}

for col_name, group, readable, severity, condition_fn in RULES:
    try:
        result = condition_fn(df)
        df[col_name] = result.astype(int)
    except Exception as e:
        print(f"  WARNING: {col_name} failed ({e}), defaulting to 0")
        df[col_name] = 0
    rule_columns.append(col_name)
    severity_map[col_name] = severity

# Count triggers per rule
print(f"\n{'Rule':<42s} {'Sev':>4s} {'Triggered':>10s} {'%':>7s}")
print("-" * 68)
for col_name, group, readable, severity, _ in RULES:
    trig = df[col_name].sum()
    pct = trig / len(df) * 100
    sev_icon = {3: "H", 2: "M", 1: "L"}[severity]
    if trig > 0:
        print(f"  {col_name:<40s} [{sev_icon}] {trig:>10,} {pct:>6.2f}%")

total_triggers = sum(df[c].sum() for c in rule_columns)
print(f"\n  Total rule triggers: {total_triggers:,}")



Executing all 50 rules...

Rule                                        Sev  Triggered       %
--------------------------------------------------------------------
  rule_large_cash_deposit                  [H]     10,458   3.20%
  rule_cash_just_below_threshold           [H]      4,851   1.48%
  rule_high_value_transfer                 [M]     56,392  17.24%
  rule_round_amount_large                  [L]          1   0.00%
  rule_night_transaction                   [M]     32,534   9.95%
  rule_weekend_high_value                  [M]     39,066  11.95%
  rule_dormant_account_activation          [H]         54   0.02%
  rule_high_acct_volume_24hr               [M]      8,336   2.55%
  rule_high_acct_volume_7d                 [M]     15,341   4.69%
  rule_amount_spike_30d                    [H]     12,557   3.84%
  rule_high_cust_velocity_1hr              [H]      9,033   2.76%
  rule_high_cust_velocity_24hr             [M]      7,930   2.42%
  rule_high_cust_volume_24hr               [M

## 8 -- Compute Composite Score & Alert Level


In [8]:
# Composite score = sum of (rule_flag * severity) across all 50 rules
df["rule_score"] = sum(df[col] * severity_map[col] for col in rule_columns)

# Rules triggered (comma-separated list of triggered rule names)
def triggered_list(row):
    triggered = [col for col in rule_columns if row[col] == 1]
    return "; ".join(triggered) if triggered else ""

df["rules_triggered"] = df.apply(triggered_list, axis=1)

# Count of rules triggered
df["rules_triggered_count"] = sum(df[col] for col in rule_columns)

# Alert level
def alert_level(score):
    if score >= 9:
        return "Critical"
    elif score >= 6:
        return "High"
    elif score >= 3:
        return "Medium"
    elif score >= 1:
        return "Low"
    return "None"

df["alert_level"] = df["rule_score"].apply(alert_level)

# Summary
print("COMPOSITE SCORING COMPLETE")
print(f"\n{'Alert Level':<12s} {'Transactions':>12s} {'%':>7s}")
print("-" * 35)
for level in ["Critical", "High", "Medium", "Low", "None"]:
    cnt = (df["alert_level"] == level).sum()
    print(f"  {level:<10s} {cnt:>12,} {cnt/len(df)*100:>6.2f}%")

print(f"\nScore distribution:")
print(f"  Min:    {df['rule_score'].min()}")
print(f"  Median: {df['rule_score'].median():.0f}")
print(f"  Mean:   {df['rule_score'].mean():.2f}")
print(f"  Max:    {df['rule_score'].max()}")
print(f"  Std:    {df['rule_score'].std():.2f}")



COMPOSITE SCORING COMPLETE

Alert Level  Transactions       %
-----------------------------------
  Critical         21,541   6.59%
  High             30,103   9.21%
  Medium           75,565  23.11%
  Low              61,480  18.80%
  None            138,336  42.30%

Score distribution:
  Min:    0
  Median: 2
  Mean:   2.57
  Max:    29
  Std:    3.10


## 9 -- Cross-Reference with Typology Labels


In [9]:
# If typology labels exist, show how rules correlate with detected typologies
if "is_aml" in df.columns:
    aml = df[df["is_aml"] == 1]
    clean = df[df["is_aml"] == 0]

    print("Rule triggering: AML vs Clean transactions\n")
    print(f"{'Rule':<42s} {'AML %':>7s} {'Clean %':>8s} {'Lift':>6s}")
    print("-" * 68)

    for col_name, _, _, sev, _ in RULES:
        aml_rate = aml[col_name].mean() * 100 if len(aml) > 0 else 0
        clean_rate = clean[col_name].mean() * 100 if len(clean) > 0 else 0
        lift = aml_rate / max(clean_rate, 0.001)
        if aml_rate > 0 or clean_rate > 0.1:
            print(f"  {col_name:<40s} {aml_rate:>6.2f}% {clean_rate:>7.2f}% {lift:>5.1f}x")

    print(f"\nAverage rule score:")
    print(f"  AML transactions:   {aml['rule_score'].mean():.2f}")
    print(f"  Clean transactions: {clean['rule_score'].mean():.2f}")

    print(f"\nAlert level distribution for AML-labeled:")
    for level in ["Critical","High","Medium","Low","None"]:
        cnt = (aml["alert_level"] == level).sum()
        print(f"  {level:<10s} {cnt:>8,} ({cnt/max(len(aml),1)*100:.1f}%)")
else:
    print("No 'is_aml' column found -- skipping typology cross-reference")



Rule triggering: AML vs Clean transactions

Rule                                         AML %  Clean %   Lift
--------------------------------------------------------------------
  rule_large_cash_deposit                    0.11%    3.42%   0.0x
  rule_cash_just_below_threshold            12.64%    0.67%  19.0x
  rule_high_value_transfer                  31.98%   16.16%   2.0x
  rule_round_amount_large                    0.00%    0.00%   4.5x
  rule_night_transaction                     0.00%   10.68%   0.0x
  rule_weekend_high_value                   14.25%   11.78%   1.2x
  rule_dormant_account_activation            0.00%    0.02%   0.3x
  rule_high_acct_volume_24hr                 5.80%    2.31%   2.5x
  rule_high_acct_volume_7d                   4.76%    4.69%   1.0x
  rule_amount_spike_30d                     13.56%    3.13%   4.3x
  rule_high_cust_velocity_1hr                2.53%    2.78%   0.9x
  rule_high_cust_velocity_24hr               2.23%    2.44%   0.9x
  rule_high_cust

## 10 -- Export Results


In [10]:
# Drop internal working columns
internal = [c for c in df.columns if c.startswith("_")]
df_out = df.drop(columns=internal)

# Full scored dataset (parquet)
path_full = os.path.join(OUTPUT_DIR, "stg_transactions_rules.parquet")
df_out.to_parquet(path_full, index=False, compression='snappy')
print(f"Full scored dataset: {path_full}")
print(f"  {len(df_out):,} rows x {len(df_out.columns)} columns")
print(f"  Size: {os.path.getsize(path_full) / (1024*1024):.2f} MB")

# High-alert subset (parquet)
df_alerts = df_out[df_out["alert_level"].isin(["Critical","High","Medium"])]
path_alerts = os.path.join(OUTPUT_DIR, "alert_transactions.parquet")
df_alerts.to_parquet(path_alerts, index=False, compression='snappy')
print(f"\nAlert transactions (Medium+): {path_alerts}")
print(f"  {len(df_alerts):,} rows")

# Rule trigger summary
summary_rows = []
for col_name, group, readable, severity, _ in RULES:
    trig = df[col_name].sum()
    summary_rows.append({
        "rule_column": col_name, "group": group,
        "readable_name": readable, "severity": severity,
        "transactions_triggered": int(trig),
        "pct_of_total": round(trig / len(df) * 100, 4),
    })
summary_df = pd.DataFrame(summary_rows).sort_values(["severity","transactions_triggered"], ascending=[False, False])
path_summary = os.path.join(OUTPUT_DIR, "rule_trigger_summary.csv")
summary_df.to_csv(path_summary, index=False)
print(f"\nRule summary: {path_summary}")
print(summary_df.to_string(index=False))

print(f"\n-- File Sizes --")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fn)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  {fn:45s} {size_mb:>8.2f} MB")

print(f"\n{'='*60}")
print(f"RULES ENGINE COMPLETE")
print(f"All outputs: {os.path.abspath(OUTPUT_DIR)}/")
print(f"{'='*60}")

Full scored dataset: ../outputs_updated\stg_transactions_rules.parquet
  327,025 rows x 151 columns
  Size: 25.88 MB

Alert transactions (Medium+): ../outputs_updated\alert_transactions.parquet
  127,209 rows

Rule summary: ../outputs_updated\rule_trigger_summary.csv
                     rule_column                 group              readable_name  severity  transactions_triggered  pct_of_total
    rule_very_high_risk_customer         KYC / Account    Very High Risk Customer         3                   37248       11.3900
        rule_multiple_small_cash           Structuring        Multiple Small Cash         3                   18325        5.6035
           rule_amount_spike_30d      Account Velocity           Amount Spike 30d         3                   12557        3.8398
         rule_large_cash_deposit                Amount         Large Cash Deposit         3                   10458        3.1979
     rule_high_cust_velocity_1hr     Customer Velocity     High Cust Velocity 1hr 

In [ ]:
# import pandas as pd
# txns = pd.read_parquet(r"C:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\stg_transactions_rules.parquet")
# txns.head().to_csv("txns_check.csv", index=False)